In [38]:
from pathlib import Path
import pandas as pd
import re
from typing import List, Dict

In [39]:
BASE_DIR = Path.cwd().parent
KNOWLEDGE_BASE_DIR = BASE_DIR / "knowledge_base"

print("BASE_DIR:", BASE_DIR)
print("KNOWLEDGE_BASE_DIR:", KNOWLEDGE_BASE_DIR)
print("Exists:", KNOWLEDGE_BASE_DIR.exists())

BASE_DIR: c:\Users\Usuario\Desktop\ProyectosPersonales\agent-architecture-advisor
KNOWLEDGE_BASE_DIR: c:\Users\Usuario\Desktop\ProyectosPersonales\agent-architecture-advisor\knowledge_base
Exists: True


In [40]:
md_files = sorted(KNOWLEDGE_BASE_DIR.rglob("*.md"))

print(f"Markdown files found: {len(md_files)}")

for file in md_files:
    print(file.relative_to(BASE_DIR))

Markdown files found: 23
knowledge_base\cloud_services\aws\amazon_s3_document_storage.md
knowledge_base\cloud_services\aws\aws_agentic_app.md
knowledge_base\cloud_services\aws\aws_ecs_fargate.md
knowledge_base\cloud_services\aws\aws_lambda_event_ingestion.md
knowledge_base\cloud_services\aws\aws_serverless_app.md
knowledge_base\cloud_services\azure\azure_agentic_app.md
knowledge_base\cloud_services\azure\azure_blob_storage.md
knowledge_base\cloud_services\azure\azure_container_apps.md
knowledge_base\cloud_services\azure\azure_functions_event_ingestion.md
knowledge_base\cloud_services\azure\azure_serverless_app.md
knowledge_base\decisions\aws_lambda_for_event_ingestion.md
knowledge_base\decisions\azure_ai_search_for_enterprise_rag.md
knowledge_base\decisions\azure_container_apps_for_agent_deployment.md
knowledge_base\decisions\bedrock_knowledge_bases_for_aws_rag.md
knowledge_base\decisions\ecs_fargate_for_agent_deployment.md
knowledge_base\decisions\qdrant_for_local_mvp.md
knowledge_bas

In [41]:
def infer_metadata_from_path(file_path: Path) -> Dict:
    """
    Infer metadata based on the file location inside knowledge_base.
    """
    parts = [part.lower() for part in file_path.parts]
    filename = file_path.stem.lower()

    provider = "unknown"
    document_type = "unknown"

    if "azure" in parts or filename.startswith("azure_"):
        provider = "azure"
    elif (
        "aws" in parts
        or filename.startswith("aws_")
        or filename.startswith("amazon_")
        or "bedrock" in filename
    ):
        provider = "aws"
    else:
        provider = "neutral"

    if "cloud_services" in parts:
        document_type = "service_reference"

    elif "patterns" in parts:
        document_type = "architecture_pattern"
        provider = "neutral"

    elif "project_cases" in parts:
        document_type = "project_case"

        if "azure" in filename:
            provider = "azure"
        elif "aws" in filename:
            provider = "aws"
        else:
            provider = "neutral"

    elif "decisions" in parts:
        document_type = "decision_record"

        if "azure" in filename:
            provider = "azure"
        elif (
            "aws" in filename
            or "bedrock" in filename
            or "textract" in filename
            or "lambda" in filename
            or "ecs" in filename
        ):
            provider = "aws"
        elif "qdrant" in filename:
            provider = "neutral"

    elif "azure" in parts or "aws" in parts:
        document_type = "cloud_reference"

    return {
        "source_file": file_path.name,
        "source_path": str(file_path),
        "provider": provider,
        "document_type": document_type,
    }

In [42]:
def clean_text(text: str) -> str:
    """
    Light cleaning while preserving Markdown structure.
    """
    text = text.replace("\r\n", "\n")
    text = re.sub(r"\n{3,}", "\n\n", text)
    return text.strip()


def read_markdown_file(file_path: Path) -> Dict:
    text = file_path.read_text(encoding="utf-8")
    metadata = infer_metadata_from_path(file_path)

    return {
        **metadata,
        "document_title": extract_document_title(text, file_path),
        "text": text,
        "clean_text": clean_text(text),
        "num_characters": len(text),
    }


def extract_document_title(text: str, file_path: Path) -> str:
    """
    Extract first H1 title from Markdown.
    Fallback to filename.
    """
    for line in text.splitlines():
        if line.startswith("# "):
            return line.replace("# ", "").strip()

    return file_path.stem.replace("_", " ").title()

In [43]:
documents = [read_markdown_file(file) for file in md_files]

df_docs = pd.DataFrame(documents)

df_docs[
    [
        "source_file",
        "document_title",
        "provider",
        "document_type",
        "num_characters",
    ]
]

,source_file,document_title,provider,document_type,num_characters
0,amazon_s3_document_storage.md,Amazon S3 for Document Storage,aws,service_reference,2036
1,aws_agentic_app.md,AWS Solution: Multi-Agent Architecture Advisor...,aws,service_reference,3786
2,aws_ecs_fargate.md,AWS ECS Fargate,aws,service_reference,1965
3,aws_lambda_event_ingestion.md,AWS Lambda for Event-Driven Document Ingestion,aws,service_reference,2069
4,aws_serverless_app.md,AWS Solution: Serverless AI Architecture Advis...,aws,service_reference,2743
5,azure_agentic_app.md,Azure Solution: Multi-Agent Architecture Advisor,azure,service_reference,4107
6,azure_blob_storage.md,Azure Blob Storage,azure,service_reference,2074
7,azure_container_apps.md,Azure Container Apps,azure,service_reference,2096
8,azure_functions_event_ingestion.md,Azure Functions for Event-Driven Document Inge...,azure,service_reference,2299
9,azure_serverless_app.md,Azure Solution: Serverless AI Architecture Adv...,azure,service_reference,2663


In [44]:
def parse_markdown_sections(text: str) -> List[Dict]:
    """
    Parse Markdown into sections using headings.
    Each section keeps its heading path.
    """
    lines = text.splitlines()

    sections = []
    current_heading_stack = []
    current_content = []
    current_section_title = "Document Introduction"
    current_level = 0

    def flush_section():
        if current_content:
            section_text = "\n".join(current_content).strip()

            if section_text:
                sections.append({
                    "section_title": current_section_title,
                    "section_path": " > ".join(current_heading_stack) if current_heading_stack else current_section_title,
                    "section_level": current_level,
                    "section_text": section_text,
                })

    for line in lines:
        heading_match = re.match(r"^(#{1,6})\s+(.*)", line)

        if heading_match:
            flush_section()

            level = len(heading_match.group(1))
            title = heading_match.group(2).strip()

            while current_heading_stack and len(current_heading_stack) >= level:
                current_heading_stack.pop()

            current_heading_stack.append(title)
            current_section_title = title
            current_level = level
            current_content = [line]

        else:
            current_content.append(line)

    flush_section()

    return sections

In [45]:
sample_text = df_docs.iloc[0]["clean_text"]
sample_sections = parse_markdown_sections(sample_text)

print(f"Sections found: {len(sample_sections)}")

for section in sample_sections[:5]:
    print("=" * 100)
    print("SECTION PATH:", section["section_path"])
    print(section["section_text"][:500])

Sections found: 10
SECTION PATH: Amazon S3 for Document Storage
# Amazon S3 for Document Storage
SECTION PATH: Amazon S3 for Document Storage > Purpose
## Purpose

Amazon S3 is an object storage service used to store unstructured data such as documents, PDFs, JSON files, logs, images and extracted text.
SECTION PATH: Amazon S3 for Document Storage > When to use this service
## When to use this service

Use Amazon S3 when a project requires:
- Storing raw documents.
- Organizing documents by project, customer, department or date.
- Triggering ingestion pipelines when new files arrive.
- Keeping original files before indexing.
- Building an AWS-native RAG architecture.
SECTION PATH: Amazon S3 for Document Storage > Typical usage in RAG architectures
## Typical usage in RAG architectures

In AWS RAG systems, Amazon S3 commonly acts as the document source layer.

Typical files:
- PDFs.
- Word documents.
- Markdown files.
- JSON exports.
- Technical documents.
- Project updates.
- Meeting n

In [46]:
def split_long_text(
    text: str,
    chunk_size: int = 1200,
    chunk_overlap: int = 200
) -> List[str]:
    """
    Split long section text into overlapping chunks.
    Used only when a Markdown section is too long.
    """
    if len(text) <= chunk_size:
        return [text.strip()]

    if chunk_size <= chunk_overlap:
        raise ValueError("chunk_size must be greater than chunk_overlap")

    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()

        if chunk:
            chunks.append(chunk)

        start += chunk_size - chunk_overlap

    return chunks

In [47]:
def slugify(text: str) -> str:
    """
    Create simple slug for IDs.
    """
    text = text.lower()
    text = re.sub(r"[^a-z0-9]+", "_", text)
    text = re.sub(r"_+", "_", text)
    return text.strip("_")


def build_contextualized_text(
    document_title: str,
    provider: str,
    document_type: str,
    source_file: str,
    section_path: str,
    chunk_text: str
) -> str:
    """
    Build the text that will be embedded and later passed to agents.
    """
    return f"""Document: {document_title}
Provider: {provider}
Document type: {document_type}
Source file: {source_file}
Section: {section_path}

{chunk_text}""".strip()


chunk_rows = []
context_counter = 1

for _, doc in df_docs.iterrows():
    sections = parse_markdown_sections(doc["clean_text"])

    for section_idx, section in enumerate(sections):
        section_chunks = split_long_text(
            section["section_text"],
            chunk_size=1200,
            chunk_overlap=200
        )

        for chunk_idx, chunk_text in enumerate(section_chunks):
            section_slug = slugify(section["section_path"])

            context_id = f"CTX-{context_counter:04d}"
            context_counter += 1

            chunk_id = (
                f"{doc['provider']}::"
                f"{doc['document_type']}::"
                f"{doc['source_file']}::"
                f"{section_slug}::"
                f"chunk_{chunk_idx}"
            )

            contextualized_chunk_text = build_contextualized_text(
                document_title=doc["document_title"],
                provider=doc["provider"],
                document_type=doc["document_type"],
                source_file=doc["source_file"],
                section_path=section["section_path"],
                chunk_text=chunk_text,
            )

            chunk_rows.append({
                "context_id": context_id,
                "chunk_id": chunk_id,
                "source_file": doc["source_file"],
                "source_path": doc["source_path"],
                "document_title": doc["document_title"],
                "provider": doc["provider"],
                "document_type": doc["document_type"],
                "section_title": section["section_title"],
                "section_path": section["section_path"],
                "section_level": section["section_level"],
                "section_index": section_idx,
                "chunk_index": chunk_idx,
                "chunk_text": chunk_text,
                "contextualized_chunk_text": contextualized_chunk_text,
                "chunk_num_characters": len(chunk_text),
                "contextualized_num_characters": len(contextualized_chunk_text),
            })

df_chunks = pd.DataFrame(chunk_rows)

df_chunks.head()

,context_id,chunk_id,source_file,source_path,document_title,provider,document_type,section_title,section_path,section_level,section_index,chunk_index,chunk_text,contextualized_chunk_text,chunk_num_characters,contextualized_num_characters
0,CTX-0001,aws::service_reference::amazon_s3_document_sto...,amazon_s3_document_storage.md,c:\Users\Usuario\Desktop\ProyectosPersonales\a...,Amazon S3 for Document Storage,aws,service_reference,Amazon S3 for Document Storage,Amazon S3 for Document Storage,1,0,0,# Amazon S3 for Document Storage,Document: Amazon S3 for Document Storage\nProv...,32,204
1,CTX-0002,aws::service_reference::amazon_s3_document_sto...,amazon_s3_document_storage.md,c:\Users\Usuario\Desktop\ProyectosPersonales\a...,Amazon S3 for Document Storage,aws,service_reference,Purpose,Amazon S3 for Document Storage > Purpose,2,1,0,## Purpose\n\nAmazon S3 is an object storage s...,Document: Amazon S3 for Document Storage\nProv...,152,334
2,CTX-0003,aws::service_reference::amazon_s3_document_sto...,amazon_s3_document_storage.md,c:\Users\Usuario\Desktop\ProyectosPersonales\a...,Amazon S3 for Document Storage,aws,service_reference,When to use this service,Amazon S3 for Document Storage > When to use t...,2,2,0,## When to use this service\n\nUse Amazon S3 w...,Document: Amazon S3 for Document Storage\nProv...,298,497
3,CTX-0004,aws::service_reference::amazon_s3_document_sto...,amazon_s3_document_storage.md,c:\Users\Usuario\Desktop\ProyectosPersonales\a...,Amazon S3 for Document Storage,aws,service_reference,Typical usage in RAG architectures,Amazon S3 for Document Storage > Typical usage...,2,3,0,## Typical usage in RAG architectures\n\nIn AW...,Document: Amazon S3 for Document Storage\nProv...,247,456
4,CTX-0005,aws::service_reference::amazon_s3_document_sto...,amazon_s3_document_storage.md,c:\Users\Usuario\Desktop\ProyectosPersonales\a...,Amazon S3 for Document Storage,aws,service_reference,Role in document ingestion,Amazon S3 for Document Storage > Role in docum...,2,4,0,## Role in document ingestion\n\nAmazon S3 can...,Document: Amazon S3 for Document Storage\nProv...,322,523


In [48]:
print("Documents:", len(df_docs))
print("Chunks:", len(df_chunks))

df_chunks.groupby(["provider", "document_type"]).size().reset_index(name="num_chunks")

Documents: 23
Chunks: 298


,provider,document_type,num_chunks
0,aws,decision_record,45
1,aws,service_reference,48
2,azure,decision_record,30
3,azure,service_reference,49
4,neutral,architecture_pattern,54
5,neutral,decision_record,15
6,neutral,project_case,57


In [49]:
sample = df_chunks.iloc[0]

print("Context ID:", sample["context_id"])
print("=" * 120)
print(sample["contextualized_chunk_text"])

Context ID: CTX-0001
Document: Amazon S3 for Document Storage
Provider: aws
Document type: service_reference
Source file: amazon_s3_document_storage.md
Section: Amazon S3 for Document Storage

# Amazon S3 for Document Storage


In [50]:
df_chunks[
    df_chunks["source_file"].str.contains("azure_ai_search", case=False)
][
    [
        "context_id",
        "provider",
        "document_type",
        "section_path",
        "chunk_text",
    ]
]

azure_decision_chunks = df_chunks[
    (df_chunks["source_file"].str.contains("azure_ai_search", case=False)) &
    (df_chunks["section_path"].str.contains("Why", case=False))
]

for _, row in azure_decision_chunks.iterrows():
    print(row["context_id"])
    print(row["contextualized_chunk_text"])
    print("=" * 120)

CTX-0118
Document: Decision Record: Azure AI Search for Enterprise RAG
Provider: azure
Document type: decision_record
Source file: azure_ai_search_for_enterprise_rag.md
Section: Decision Record: Azure AI Search for Enterprise RAG > Why this decision was made

## Why this decision was made

Azure AI Search was chosen because it provides a managed retrieval service within the Azure ecosystem. It supports keyword search, vector search, hybrid search, semantic ranking and metadata filtering. These capabilities made it suitable for enterprise RAG systems that need to retrieve grounded context from private documents.


In [51]:
PROCESSED_DIR = BASE_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

docs_output_path = PROCESSED_DIR / "knowledge_base_docs.parquet"
chunks_output_path = PROCESSED_DIR / "knowledge_base_chunks.parquet"

df_docs.to_parquet(docs_output_path, index=False)
df_chunks.to_parquet(chunks_output_path, index=False)

print("Saved docs:", docs_output_path)
print("Saved chunks:", chunks_output_path)

Saved docs: c:\Users\Usuario\Desktop\ProyectosPersonales\agent-architecture-advisor\data\processed\knowledge_base_docs.parquet
Saved chunks: c:\Users\Usuario\Desktop\ProyectosPersonales\agent-architecture-advisor\data\processed\knowledge_base_chunks.parquet


In [52]:
loaded_chunks = pd.read_parquet(chunks_output_path)

loaded_chunks[
    [
        "context_id",
        "provider",
        "document_type",
        "source_file",
        "section_path",
    ]
].head(10)

,context_id,provider,document_type,source_file,section_path
0,CTX-0001,aws,service_reference,amazon_s3_document_storage.md,Amazon S3 for Document Storage
1,CTX-0002,aws,service_reference,amazon_s3_document_storage.md,Amazon S3 for Document Storage > Purpose
2,CTX-0003,aws,service_reference,amazon_s3_document_storage.md,Amazon S3 for Document Storage > When to use t...
3,CTX-0004,aws,service_reference,amazon_s3_document_storage.md,Amazon S3 for Document Storage > Typical usage...
4,CTX-0005,aws,service_reference,amazon_s3_document_storage.md,Amazon S3 for Document Storage > Role in docum...
5,CTX-0006,aws,service_reference,amazon_s3_document_storage.md,Amazon S3 for Document Storage > Example organ...
6,CTX-0007,aws,service_reference,amazon_s3_document_storage.md,Amazon S3 for Document Storage > Pros
7,CTX-0008,aws,service_reference,amazon_s3_document_storage.md,Amazon S3 for Document Storage > Cons
8,CTX-0009,aws,service_reference,amazon_s3_document_storage.md,Amazon S3 for Document Storage > Best suited for
9,CTX-0010,aws,service_reference,amazon_s3_document_storage.md,Amazon S3 for Document Storage > Not ideal for
